In [ ]:
# declare a list tasks whose products you want to use as inputs
upstream = None
product = None
mcleish24_tables4 = None
virus_hits = None

In [9]:
import pandas as pd
import requests
import json
import seaborn as sns 
import random
import datetime

In [5]:
def timestamp_code():
    """
    This function generates a time stamp that goes
    to microsecond level, and it adds a random code
    just in case
    """
    now = datetime.datetime.now()
    timestamp = now.strftime("%Y%m%d%H%M%S%f")
    random.seed(timestamp[-4:])
    random_code = "{:03d}".format(random.randint(0, 100))
    return timestamp +  random_code

print(timestamp_code())
print(timestamp_code())
for i in range(1000):
    if timestamp_code() == timestamp_code():
        raise RuntimeError("the function did not work")

20251006111415703190030
20251006111415703363093


In [ ]:
virus = pd.read_csv(mcleish24_tables4, sep=';')
virus

,OTU name,Family,NCBI_title,OTU_reference,Genome,%_ID_mean,%_ID_min,%_ID_max,BLAST_matches,NCBI_accession
0,Alfamovirus 1,Bromoviridae,Alfalfa mosaic virus RNA 1,AMV1,(+)ssRNA,96.6,94.4,99.2,4,NC_001495
1,Alfamovirus 1,Bromoviridae,Alfalfa mosaic virus RNA 2,AMV2,(+)ssRNA,94.1,94.1,94.1,2,NC_002024
2,Alphaendornavirus 1,Endornaviridae,Cucumis melo endornavirus,CmEV,dsRNA,96.5,87.9,100.0,4174,NC_029064
3,Alphaendornavirus 2,Endornaviridae,Bell pepper endornavirus,BPEV,dsRNA,100.0,100.0,100.0,2,NC_015781
4,Alphaendornavirus 3,Endornaviridae,Hordeum vulgare endornavirus,HvEV,dsRNA,96.6,87.2,100.0,4002,NC_028949
...,...,...,...,...,...,...,...,...,...,...
172,Comovirus,Secoviridae,Arabidopsis latent virus-1 RNA1,ALaV1,(+)ssRNA,99.5,97.3,100.0,52,MH899120
173,Comovirus,Secoviridae,Arabidopsis latent virus-1 RNA2,ALaV2,(+)ssRNA,97.6,94.0,100.0,30,MH899121
174,Varicosavirus 1,Rhabdoviridae,Lettuce big-vein associated virus RNA 1,LBVaV1,(-)ssRNA,95.3,93.3,97.3,12,NC_011558
175,Varicosavirus 1,Rhabdoviridae,Lettuce big-vein associated virus RNA 2,LBVaV2,(-)ssRNA,94.9,90.6,98.0,32,NC_011568


The file provided by Fernando requires some actions to turn it from its matrix form into a list of Library - Otu, which will be much easier to analyze later.

In [11]:
def extract_taxonomy_from_gene_records(x):
    url = "https://api.ncbi.nlm.nih.gov/datasets/v2"
    url = f'{url}/gene/accession/{x}/dataset_report'.replace(' ', '%20')
    r = requests.get(url)
    
    try: 
        r.raise_for_status()
    except requests.HTTPError:
        return pd.NA
    try:
        u = str(r.json()['reports'][0]['gene']['tax_id'])
    except KeyError:
        return pd.NA
    except IndexError:
        return pd.NA
    return u

extract_taxonomy_from_gene_records("NC_030889") # 1849335

'1849335'

In [12]:
virus['OTU_taxid'] = virus['NCBI_accession'].apply(extract_taxonomy_from_gene_records)

In [13]:

virus_subset = virus[['OTU_taxid', 'NCBI_title', 'OTU_reference', 'NCBI_accession']].rename(columns={'OTU_taxid': 'taxid', 'NCBI_title': 'Scientific_name', 'OTU_reference': 'label'})
virus_subset['Taxon_code'] = [timestamp_code() for _ in range(len(virus_subset))]
virus_subset

,taxid,Scientific_name,label,NCBI_accession,Taxon_code
0,12321,Alfalfa mosaic virus RNA 1,AMV1,NC_001495,20251006111738644281055
1,12321,Alfalfa mosaic virus RNA 2,AMV2,NC_002024,20251006111738644349034
2,1776177,Cucumis melo endornavirus,CmEV,NC_029064,20251006111738644380057
3,<NA>,Bell pepper endornavirus,BPEV,NC_015781,20251006111738644407045
4,1774276,Hordeum vulgare endornavirus,HvEV,NC_028949,20251006111738644433001
...,...,...,...,...,...
172,<NA>,Arabidopsis latent virus-1 RNA1,ALaV1,MH899120,20251006111738648642089
173,<NA>,Arabidopsis latent virus-1 RNA2,ALaV2,MH899121,20251006111738648667061
174,1985698,Lettuce big-vein associated virus RNA 1,LBVaV1,NC_011558,20251006111738648692070
175,1985698,Lettuce big-vein associated virus RNA 2,LBVaV2,NC_011568,20251006111738648716092


In [ ]:
hits = pd.read_csv(virus_hits, sep=';')
hits = hits.melt(
    id_vars=['Library', 'Host', 'Host_family', 'Habitat', 'Site', 'Collection'],
    value_vars=hits.columns[6:], 
).rename(columns={'variable': "OTU", 'value': "hit"}).query('hit == 1').copy()
hits['Collection'] = hits['Collection'].apply(lambda x: x.split("_")[0])
hits[['Library', 'Host', 'Host_family', 'Habitat', 'Site', 'Collection', 'OTU']]


,Library,Host,Host_family,Habitat,Site,Collection,OTU
74,PV091,Galium verum,Rubiaceae,Oak,Q4,Q4P,AEV1
332,PV059,Tragopogon sp,Asteraceae,Wasteland,E4,E4P,AEYV
425,PV159,Carduus bourgeanus,Asteraceae,Edge,L1,L1V,AEYV
448,PV182,Lactuca serriola,Asteraceae,Edge,L3,L3V,AEYV
537,PV498,Rubia peregrina,Rubiaceae,Crop,H3,H3P,AEYV
...,...,...,...,...,...,...,...
44458,PV172,Picris echioides,Asteraceae,Edge,L2,L2V,YSV
44487,PV201,Convolvulus arvensis,Convolvulaceae,Edge,L1,L1F,YSV
44625,PV047,Zea mays,Poaceae,Crop,Z2,Z2V,ZYMV
44638,PV061,Cucumis melo,Cucurbitaceae,Crop,M1,M5V,ZYMV


In [21]:
rename_map = pd.read_csv("input/mapping-otus-curated.csv", sep="\t", index_col=0)[['otu', 'hit']].set_index('otu').to_dict()['hit']
hits['OTU_rn'] = hits['OTU'].apply(lambda x: rename_map[x])
hits[hits['OTU_rn'].isna()][['OTU']].value_counts()

Series([], Name: count, dtype: int64)

In [22]:
hits = pd.merge(hits.dropna(subset=['OTU_rn'])[['Library', 'OTU_rn']], virus_subset[['label', 'Taxon_code']], how='left', right_on='label', left_on='OTU_rn')[['Library', 'Taxon_code']]
# hits[hits['OTU name'].isna()][['OTU']].value_counts()

In [23]:
hits

,Library,Taxon_code
0,PV091,20251006111738645711014
1,PV059,20251006111738646382087
2,PV159,20251006111738646382087
3,PV182,20251006111738646382087
4,PV498,20251006111738646382087
...,...,...
1611,PV172,20251006111738644733078
1612,PV201,20251006111738644733078
1613,PV047,20251006111738647525071
1614,PV061,20251006111738647525071


In [24]:
hits['Hit_code'] = [timestamp_code() for _ in range(len(hits))]
hits[['Hit_code', 'Library', 'Taxon_code']]

,Hit_code,Library,Taxon_code
0,20251006112449250133083,PV091,20251006111738645711014
1,20251006112449250224023,PV059,20251006111738646382087
2,20251006112449250255026,PV159,20251006111738646382087
3,20251006112449250282001,PV182,20251006111738646382087
4,20251006112449250308083,PV498,20251006111738646382087
...,...,...,...
1611,20251006112449288417003,PV172,20251006111738644733078
1612,20251006112449288437037,PV201,20251006111738644733078
1613,20251006112449288456078,PV047,20251006111738647525071
1614,20251006112449288476073,PV061,20251006111738647525071


In [25]:
clean_dict = lambda d: {k: d[k] for k in d if not pd.isna(k)}

In [31]:
hits.query('Hit_code == "000000000111"')

,Library,Taxon_code,Hit_code
111,PV176,000000000044,000000000111


In [ ]:
virus_library = dict(
    virus=list(map(clean_dict, virus_subset.to_dict(orient='records'))),
    hits=list(map(clean_dict, hits.to_dict(orient='records')))
)
with open(product['data'], "w") as f:
    json.dump(virus_library, f, indent=4, sort_keys=True)